# Day 06 · Vintage — Surface Lines, 1929

**Question.** How much of today's CTA bus network still runs where the Chicago Surface Lines' streetcars ran?

The Chicago Surface Lines (CSL) ran the city's streetcars, and some buses, from 1914 until the CTA took it
over on 1 October 1947; the last streetcar ran on 21 June 1958. The CTA now runs buses on many of the same
streets, and every bus has a ramp. This notebook measures how much of the CSL network, as the CSL's own
1929 system map shows it, still has a CTA bus. 1929 is the date of the map, not of the network.

**Approach.** Georeference the Surface Lines' own 1929 system map (a public-domain scan) onto the
atlas's address grid, read its red lines off every OpenStreetMap street, and compare with today's CTA
bus routes (the Day 02 network). Count street-miles: on both, 1929 only, today only.

| Section | What it does |
| --- | --- |
| 1. Setup | Imports, settings, paths, constants |
| 2. Pull | The scan (IIIF, full size), OSM streets, city limits, today's bus routes |
| 3. Validate | Sizes and counts |
| 4. Georeference | Margin address ticks, then line snapping to OSM arterials; the Central District inset |
| 5. Classify | Red ink along each street; today's routes on each street |
| 6. Analyze | Street-miles, shares, rings from the Loop; the finding |
| 7. QA | Registration residuals, totals, a random sample sheet for a check by eye |
| 8. Export | `output/plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import subprocess
from datetime import date
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import requests
import shapely
from PIL import Image
from scipy.interpolate import RBFInterpolator
from scipy.ndimage import uniform_filter
from scipy.spatial import cKDTree

from atlas import Ledger, to_address, walk, write_plate

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day06')
Image.MAX_IMAGE_PIXELS = None

In [ ]:
# Run settings ---------------------------------------------------------------------------------
DAY = 6
SCAN_ID = 'p16022coll205:627'   # UMN Borchert Map Library, System of Chicago Surface Lines (1929)
SCAN_RECORD = 'https://umedia.lib.umn.edu/item/p16022coll205:627'
SCAN_IIIF = 'https://cdm16022.contentdm.oclc.org/iiif/2/p16022coll205:627'
OSHER_RECORD = 'https://oshermaps.org/map/54538.0001'
CTA_ACCESS_URL = 'https://www.transitchicago.com/accessibility/features/'

# Paths: the notebook runs from its own folder (days/day06-vintage/)
DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
PROC_DIR = DAY_DIR / 'data' / 'processed'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
OSM_DIR = REPO / walk.OSM_DIR
BOUNDARY = REPO / 'reference' / 'city-boundary.json'
ROUTES = REPO / 'days' / 'day02-lines' / 'output' / 'plate.json'   # today's bus network, as Plate II
assert (DAY_DIR / 'day06.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'

CRS = walk.CRS
MILE = 800                      # address numbers per mile (the first mile south of Madison has 1,200)

# Method parameters (pixels are scan pixels; 1 px is about 4.3 address units, 29 ft) -------------
SNAP_SCHEDULE = ((40, 2.0), (20, 0.5), (10, 0.2), (6, 0.1))   # search window px, field smoothing
ALONG_PX = 8          # red must run this far along the street on both sides of a point
ACROSS_PX = 10        # ... and lie within this distance across it (main map)
ACROSS_INSET_PX = 8   # the same in the Central District inset
SIDE_STREET_PX = 5    # a side street counts this much farther when two streets compete for one red line
ALONG_STREET = 200    # smoothing window along a street, address units (a quarter mile)
ON_ROUTE = 25         # a street carries a route if the route runs within this, address units, and parallel
CELL = 40             # street-miles are counted on 40-unit cells, so both roadways of a divided street count once
HALF_MILE = 400

## 2. Pull

**The map.** *System of Chicago Surface Lines, showing principal points of interest and direct lines of
transportation* (Chicago Surface Lines, 1929; 52 × 27 cm, about 1:104,500). The University of Minnesota's
Borchert Map Library serves a full-size scan (3609 × 6647 px) over IIIF; the same map is at the Osher Map
Library. The sheet reads "Copyright, 1929, by Chicago Surface Lines", so it is in the public domain in the
US since 1 January 2025. Street car lines are printed solid red, the company's few bus lines dashed red,
and the four margins carry Chicago street numbers.

**Streets.** OSM, the drivable named network from the newest Geofabrik extract (as Day 04), as an
undirected graph. **Today's routes** are Plate II's: CTA GTFS shapes in address units, with the second
street of each one-way pair.

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True); PROC_DIR.mkdir(parents=True, exist_ok=True)
scans = sorted(RAW_DIR.glob('csl_1929_umn_*.jpg'))
if not scans:
    r = requests.get(f'{SCAN_IIIF}/full/full/0/default.jpg', timeout=300); r.raise_for_status()
    scans = [RAW_DIR / f'csl_1929_umn_{date.today().isoformat()}.jpg']; scans[0].write_bytes(r.content)
SCAN = scans[-1]
scan_pulled = date.fromisoformat(SCAN.stem.rsplit('_', 1)[-1])
A = np.asarray(Image.open(SCAN).convert('RGB')).astype(int)
H, W = A.shape[:2]
log.info('Scan %s: %d x %d px', SCAN.name, W, H)

pbf, osm_date = walk.latest_pbf(OSM_DIR)
streets_pbf = PROC_DIR / f'chicago-streets-{osm_date.isoformat()}.osm.pbf'
if not streets_pbf.exists():
    box = PROC_DIR / f'chicago-{osm_date.isoformat()}.osm.pbf'
    subprocess.run(['osmium', 'extract', '-b', '-87.95,41.64,-87.52,42.03', str(pbf), '-o', str(box), '--overwrite'], check=True)
    subprocess.run(['osmium', 'tags-filter', str(box),
                    'w/highway=trunk,primary,secondary,tertiary,unclassified,residential,living_street',
                    '-o', str(streets_pbf), '--overwrite'], check=True)
graph_pkl = PROC_DIR / f'chicago-streets-{osm_date.isoformat()}.graph.pkl'
if graph_pkl.exists():
    G = pd.read_pickle(graph_pkl)
else:
    xml = PROC_DIR / f'chicago-streets-{osm_date.isoformat()}.osm'
    if not xml.exists():
        subprocess.run(['osmium', 'cat', str(streets_pbf), '-o', str(xml), '--overwrite'], check=True)
    G = ox.convert.to_undirected(ox.project_graph(ox.graph_from_xml(xml, simplify=True, retain_all=False), to_crs=CRS))
    pd.to_pickle(G, graph_pkl)

rows = []
for u, v, d in G.edges(data=True):
    geom = d.get('geometry') or shapely.LineString([(G.nodes[u]['x'], G.nodes[u]['y']), (G.nodes[v]['x'], G.nodes[v]['y'])])
    hw, nm = d.get('highway'), d.get('name')
    rows.append((hw if isinstance(hw, str) else '|'.join(hw), nm if isinstance(nm, str) else ('|'.join(nm) if nm else ''), geom))
edges = gpd.GeoDataFrame(rows, columns=['highway', 'name', 'geometry'], crs=CRS).to_crs('EPSG:4326')


def to_addr_line(g: shapely.LineString) -> shapely.LineString:
    xy = np.asarray(g.coords); x, y = to_address(xy[:, 1], xy[:, 0]); return shapely.LineString(np.c_[x, y])


st = pd.DataFrame({'highway': edges['highway'], 'name': edges['name'],
                   'geometry': [to_addr_line(g) for g in edges.geometry]})
st['len'] = shapely.length(st['geometry'].values)
mid = shapely.get_coordinates(shapely.line_interpolate_point(st['geometry'].values, 0.5, normalized=True))
st['mx'], st['my'] = mid[:, 0], mid[:, 1]

city = None
for ring in json.loads(BOUNDARY.read_text())['rings']:      # rings are drawn even-odd: outer and holes
    p = shapely.make_valid(shapely.Polygon(ring)); city = p if city is None else shapely.symmetric_difference(city, p)
st['inCity'] = shapely.contains_xy(city, st['mx'], st['my'])

day02 = json.loads(ROUTES.read_text())
route_paths = [(r['id'], np.array(r['path'], float)[:, :2]) for r in day02['routes']]
route_paths += [(r['id'], np.array(a, float)[:, :2]) for r in day02['routes'] for a in (r['alt'] or [])]
log.info('OSM %s: %d street edges (%d in the city); %d bus routes (GTFS %s)', osm_date, len(st), st['inCity'].sum(),
         len(day02['routes']), day02['gtfs']['feedDate'])

## 3. Validate

In [ ]:
assert (W, H) == (3609, 6647), 'Unexpected scan size: the margin tick tables below are in these pixels'
assert st['inCity'].sum() > 30_000
assert len(day02['routes']) == day02['summary']['routes']
print(st['highway'].value_counts().head(8).to_string())

## 4. Georeference

The 1929 map is already drawn on the address grid: its margins carry street numbers. So the fit goes
from scan pixels straight to address units, through miles (the first mile south of Madison holds 1,200
numbers, as in `atlas.grid`).

1. **Margin ticks.** Each margin label (read by eye, with its approximate pixel) is snapped to the tan
   tick mark printed beside it. East (x) is fitted from the top and bottom margins, north (y) from the
   left and right ones, each with only the terms two margins can fix.
2. **Line snapping.** Points along OSM's major streets that run north–south or east–west are moved onto
   the nearest red line across the street; smooth correction fields in x and y are fitted from the moves,
   in four passes with a shrinking search window. Half-mile arterials are 92 px apart, so a point cannot
   jump to the next arterial. (Matching tiles of the whole street grid was tried and dropped: it can slip
   by one block, 23 px.)
3. **The Central District inset** (Division to Roosevelt) is a schematic at a larger scale. It gets its
   own piecewise-linear x and y scales from street lines found in it.

In [ ]:
# Ink masks -----------------------------------------------------------------------------------------
R_, G_, B_ = A[..., 0], A[..., 1], A[..., 2]
RED = (R_ - G_ > 55) & (R_ - B_ > 60) & (R_ > 150)              # red ink (registration)
RED_EXT = RED | ((R_ - G_ > 45) & (R_ - B_ > 70) & (R_ > 105))  # plus red printed over green, which reads brown
REDS = uniform_filter(RED.astype(float), 3) > 0.34
REDS_EXT = uniform_filter(RED_EXT.astype(float), 3) > 0.34


def y_miles(y):
    y = np.asarray(y, float)
    return np.select([y >= 0, y >= -1200], [y / MILE, y / 1200], default=-1 + (y + 1200) / MILE)


def ticks(band, off, axis, frac=0.25, gap=3):
    """Centres of the tan tick marks in a margin band (columns for axis 0, rows for axis 1)."""
    r, g, b = band[..., 0], band[..., 1], band[..., 2]
    tan = (r - b > 50) & (g - b > 20) & (r > 140) & (r - g < 55)
    col = tan.sum(axis); xs = np.where(col > col.max() * frac)[0]
    groups, cur = [], [xs[0]]
    for x in xs[1:]:
        if x - cur[-1] <= gap: cur.append(x)
        else: groups.append(cur); cur = [x]
    groups.append(cur)
    return np.array([float(np.average(g, weights=col[g])) + off for g in groups])


# Margin labels: (address, approximate pixel read by eye). x: - west / + east; y: + north / - south.
TOP = [(-8400, 250), (-7200, 574), (-6555, 737), (-6400, 768), (-6000, 857), (-5600, 933), (-5501, 957), (-5440, 994),
       (-5200, 1046), (-4900, 1106), (-4800, 1131), (-4600, 1177), (-4446, 1209), (-4000, 1313), (-3400, 1451), (-3200, 1511),
       (-2833, 1575), (-2800, 1597), (-2400, 1696), (-2000, 1777), (-1800, 1843), (-1600, 1890), (-1500, 1918), (-1400, 1945),
       (-1200, 1979), (-1000, 2015), (-932, 2046), (-800, 2083), (-600, 2120), (-400, 2160), (-200, 2198), (-100, 2233), (0, 2263),
       (50, 2288), (200, 2314), (483, 2390), (800, 2464)]
BOTTOM = [(-4800, 1147), (-4600, 1201), (-4400, 1252), (-4000, 1333), (-3700, 1406), (-3600, 1438), (-3200, 1507), (-3000, 1558),
          (-2900, 1586), (-2800, 1617), (-2600, 1665), (-2400, 1708), (-2200, 1752), (-2000, 1794), (-1800, 1834), (-1700, 1862),
          (-1600, 1890), (-1400, 1919), (-1300, 1945), (-1200, 1972), (-1000, 2030), (-800, 2075), (-600, 2103), (-500, 2133),
          (-425, 2162), (-200, 2196), (-33, 2227), (0, 2258), (100, 2288), (200, 2314), (400, 2343), (600, 2386), (946, 2484),
          (1000, 2512), (1400, 2600), (1500, 2631), (1800, 2699), (2635, 2868), (3100, 2976), (3158, 3007), (3435, 3049),
          (3633, 3125), (4100, 3215)]
LEFT = [(7548, 1242), (7200, 1333), (6439, 1497), (6400, 1525), (5800, 1670), (5600, 1717), (5300, 1787), (4800, 1900),
        (4400, 2002), (4000, 2090), (3900, 2120), (3200, 2280), (2400, 2460), (2000, 2558), (1600, 2648), (1200, 2750), (800, 2833),
        (634, 2863), (440, 2893), (433, 2925), (150, 2995), (0, 3028), (-100, 3060), (-400, 3090), (-600, 3120), (-1000, 3182),
        (-1200, 3216), (-1600, 3315), (-2200, 3405), (-2500, 3480), (-3100, 3595), (-3600, 3708), (-3900, 3778), (-4700, 3970),
        (-5100, 4060), (-5500, 4152), (-5600, 4182), (-5900, 4248), (-6200, 4310), (-6300, 4340), (-6500, 4392), (-6800, 4457),
        (-7100, 4530)]
RIGHT = [(5835, 1668), (5600, 1728), (5400, 1776), (5200, 1823), (4800, 1913), (4400, 2012), (4000, 2098), (3700, 2177),
         (3400, 2240), (3200, 2288), (2800, 2385), (2200, 2523), (2000, 2565), (1900, 2595), (1600, 2660), (1400, 2710), (1200, 2760),
         (800, 2855), (530, 2920), (300, 2960), (200, 2992), (0, 3037), (-200, 3072), (-600, 3127), (-900, 3176), (-1200, 3221),
         (-1400, 3264), (-1600, 3300), (-1800, 3342), (-2100, 3381), (-2200, 3414), (-2600, 3503), (-2900, 3553), (-3100, 3597),
         (-3500, 3683), (-3800, 3753), (-4151, 3835), (-4300, 3875), (-4700, 3965), (-5100, 4067), (-5500, 4150), (-5636, 4181),
         (-5900, 4249), (-6100, 4294), (-6300, 4333), (-6400, 4367), (-6700, 4431), (-6900, 4475), (-7100, 4523), (-7300, 4566),
         (-7500, 4614), (-7900, 4702), (-8300, 4797), (-8700, 4891), (-8900, 4939), (-9100, 4982), (-9300, 5034), (-9500, 5081),
         (-9800, 5153), (-10400, 5297), (-10600, 5344), (-10800, 5394), (-11400, 5523), (-11800, 5612), (-12200, 5711),
         (-12400, 5741), (-13000, 5911), (-13558, 6036), (-13800, 6093)]
BANDS = {  # margin: (labels, detected ticks, shift of the read position, fixed coordinate of the margin)
    'top': (TOP, ticks(A[1000:1220, 200:3000], 200, 0), -3, 1110),
    'bottom': (BOTTOM, ticks(A[6180:6300, 1000:3300], 1000, 0), 3, 6240),
    'left': (LEFT, ticks(A[1150:6300, 345:385], 1150, 1), 6, 365),
    'right': (RIGHT, ticks(A[1600:6400, 3240:3280], 1600, 1), -2, 3260),
}
gcp = []
for side, (labels, det, shift, fixed) in BANDS.items():
    for addr, p in labels:
        i = np.argmin(np.abs(det - (p + shift)))
        if abs(det[i] - (p + shift)) <= 12:   # unsnapped labels are not used
            px, py = (det[i], fixed) if side in ('top', 'bottom') else (fixed, det[i])
            gcp.append(dict(side=side, axis='x' if side in ('top', 'bottom') else 'y', addr=addr, px=px, py=py,
                            miles=addr / MILE if side in ('top', 'bottom') else float(y_miles(addr))))
gcp = pd.DataFrame(gcp)
print(gcp.groupby('side').size().to_string(), f'\nsnapped {len(gcp)} of {sum(len(b[0]) for b in BANDS.values())} labels')

In [ ]:
def _n(px, py): return (np.asarray(px, float) - 1800) / 1000, (np.asarray(py, float) - 3300) / 1000
def D_east(px, py): x, y = _n(px, py); return np.column_stack([np.ones_like(x), x, x * x, x ** 3, y, x * y, x * x * y])
def D_north(px, py): x, y = _n(px, py); return np.column_stack([np.ones_like(y), y, y * y, y ** 3, x, x * y, x * y * y])

margin_rms = {}
coef = {}
for axis, D in (('x', D_east), ('y', D_north)):
    g = gcp[gcp['axis'] == axis]; keep = np.ones(len(g), bool)
    for _ in range(5):
        c, *_ = np.linalg.lstsq(D(g.px[keep], g.py[keep]), g.miles[keep], rcond=None)
        res = D(g.px, g.py) @ c - g.miles; s = np.sqrt(np.mean(res[keep] ** 2)); keep = np.abs(res) <= 2.5 * s
    coef[axis] = c; margin_rms[axis] = round(float(s * MILE), 1)
log.info('Margin fit: %d / %d ticks; RMS %s address units', (gcp.axis == 'x').sum(), (gcp.axis == 'y').sum(), margin_rms)

# Inverse (miles -> px) as a cubic in east and north miles, fitted through the forward model on a pixel grid
gx, gy = (a.ravel() for a in np.meshgrid(np.linspace(300, 3400, 70), np.linspace(1100, 6300, 100)))
ge, gn = D_east(gx, gy) @ coef['x'], D_north(gx, gy) @ coef['y']
def D_inv(e, n):
    e, n = np.asarray(e, float) / 10, np.asarray(n, float) / 10
    return np.column_stack([e ** i * n ** j for i in range(4) for j in range(4 - i)])
inv_x, *_ = np.linalg.lstsq(D_inv(ge, gn), gx, rcond=None); inv_y, *_ = np.linalg.lstsq(D_inv(ge, gn), gy, rcond=None)
assert np.hypot(D_inv(ge, gn) @ inv_x - gx, D_inv(ge, gn) @ inv_y - gy).max() < 5


class Warp:
    """Address units -> scan pixels: the margin model plus smooth x and y corrections from line snapping."""
    def __init__(self): self.fx = self.fy = None
    def __call__(self, x, y):
        e, n = np.asarray(x, float) / MILE, y_miles(y)
        px, py = D_inv(e, n) @ inv_x, D_inv(e, n) @ inv_y
        q = np.c_[px, py] / 1000
        if self.fx is not None: px = px + self.fx(q)
        if self.fy is not None: py = py + self.fy(q)
        return px, py


def snap_samples(lines, spacing=150.0, tol_deg=8.0):
    """Points every `spacing` units along streets that run north-south (v) or east-west (h)."""
    out = []
    for g in lines:
        L = g.length
        if L < spacing / 2: continue
        d = np.arange(spacing / 2, L, spacing)
        a = shapely.get_coordinates(shapely.line_interpolate_point(g, np.maximum(d - 20, 0)))
        b = shapely.get_coordinates(shapely.line_interpolate_point(g, np.minimum(d + 20, L)))
        p = shapely.get_coordinates(shapely.line_interpolate_point(g, d))
        ang = np.degrees(np.arctan2(b[:, 1] - a[:, 1], b[:, 0] - a[:, 0])) % 180
        for q, an in zip(p, ang):
            if abs(an - 90) < tol_deg: out.append((q[0], q[1], 'v'))
            elif min(an, 180 - an) < tol_deg: out.append((q[0], q[1], 'h'))
    return pd.DataFrame(out, columns=['x', 'y', 'k'])


def offsets(S, w, win, half=10, min_px=10):
    """Offset across the street (px) from each sample to the nearest red line; nan where none or two."""
    px, py = w(S.x.values, S.y.values); off = np.full(len(S), np.nan)
    for i, (x, y, k) in enumerate(zip(px, py, S.k.values)):
        x, y = int(round(x)), int(round(y))
        if not (win + 2 < x < W - win - 2 and win + 2 < y < H - win - 2): continue
        prof = REDS[y - half:y + half, x - win:x + win + 1].sum(0) if k == 'v' else REDS[y - win:y + win + 1, x - half:x + half].sum(1)
        j = int(np.argmax(prof))
        if prof[j] < min_px: continue
        q = prof.copy(); q[max(0, j - 6):j + 7] = 0
        if q.max() > 0.7 * prof[j]: continue
        idx = np.arange(max(0, j - 3), min(len(prof), j + 4))
        off[i] = np.average(idx, weights=prof[idx] + 1e-9) - win
    return px, py, off


major_lines = st.loc[st['highway'].str.contains('primary|secondary'), 'geometry'].values
S = snap_samples(major_lines)
warp = Warp(); snap_log = []
for win, sm in SNAP_SCHEDULE:
    px, py, off = offsets(S, warp, win)
    for k in 'vh':
        m = (S.k.values == k) & ~np.isnan(off); q = np.c_[px[m], py[m]] / 1000
        f = warp.fx if k == 'v' else warp.fy
        tgt = (f(q) if f is not None else 0) + off[m]; keep = np.ones(m.sum(), bool)
        for _ in range(3):
            g = RBFInterpolator(q[keep], tgt[keep], kernel='thin_plate_spline', smoothing=sm * keep.sum())
            r = g(q) - tgt; s = np.sqrt(np.mean(r[keep] ** 2)); keep = np.abs(r) < max(2.5 * s, 1.5)
        if k == 'v': warp.fx = g
        else: warp.fy = g
        snap_log.append(dict(window=win, axis=k, samples=int(m.sum()), kept=int(keep.sum()),
                             medianOffset=round(float(np.nanmedian(np.abs(off[m]))), 1), rms=round(float(s), 1)))
snap_log = pd.DataFrame(snap_log); print(snap_log.to_string(index=False))

In [ ]:
# The Central District inset: street lines found in the inset (px) against the same streets in OSM (address units)
INSET_X = [(-805, 2595.0), (-600, 2703.0), (-480, 2773.5), (-384, 2846.6), (-321, 2876.0), (-266, 2913.0), (-139, 2990.0),
           (-59, 3027.3), (0, 3063.0), (77, 3103.0)]
INSET_Y = [(-1217, 1478.5), (-1026, 1401.0), (-629, 1237.0), (-433, 1164.0), (-223, 1089.0), (-119, 1049.5), (-17, 1006.4),
           (132, 919.0), (206, 879.9), (444, 736.5), (509, 700.5), (795, 528.5), (1200, 290.3)]
INSET_AREA = (-860, 600, -1240, 1300)   # x0, x1, y0, y1 in address units, read from the inset


def piecewise(v, pairs):
    a, b = np.array(pairs, float).T; v = np.asarray(v, float); out = np.interp(v, a, b)
    lo, hi = v < a[0], v > a[-1]
    out[lo] = b[0] + (v[lo] - a[0]) * (b[1] - b[0]) / (a[1] - a[0]); out[hi] = b[-1] + (v[hi] - a[-1]) * (b[-1] - b[-2]) / (a[-1] - a[-2])
    return out


def inset_warp(x, y): return piecewise(x, INSET_X), piecewise(y, INSET_Y)


x0, x1, y0, y1 = INSET_AREA
st['inset'] = st['mx'].between(x0, x1) & st['my'].between(y0, y1)
px, py = warp(st['mx'].values, st['my'].values)
st['onMap'] = ((px > 420) & (px < 3240) & (py > 1180) & (py < 6200) & ~((px < 1100) & (py > 4560))) | st['inset']
log.info('City streets off the 1929 sheet: %.0f mi (the O\'Hare area)', st.loc[st.inCity & ~st.onMap, 'len'].sum() / MILE)

## 5. Classify

**1929.** A point of a street is on a 1929 line where red ink runs *along* the street there: red at the
point and 8 px ahead and behind, within 10 px across (8 px in the inset). A red line crossing the street
is narrower than 16 px, so it does not count. Red printed over parks and the river reads brown and is
included. Each red run goes to the nearest parallel street; side streets count 5 px farther, so a line
drawn a little off its arterial stays on it, and the two roadways of one street do not compete. Shares
are smoothed over a quarter mile along each street name; a street edge is a 1929 line at 50% or more.

**Today.** A street carries a bus where one of today's routes runs within 25 address units of it and
parallel to it (50 units caught the next residential street).

In [ ]:
def along(lines, w, mask, across, step=1.0):
    """Per line: sample points (px), unit directions, normals, and the across-street offset of the nearest red run along it."""
    offs = np.array(sorted(range(-across, across + 1), key=abs))
    at = lambda x, y: mask[np.clip(np.rint(y).astype(int), 0, H - 1), np.clip(np.rint(x).astype(int), 0, W - 1)]
    res = []
    for geom in lines:
        xy = np.asarray(geom.coords); qx, qy = w(xy[:, 0], xy[:, 1])
        ln = shapely.LineString(np.c_[qx, qy]); L = ln.length
        d = np.arange(0, L + 1e-9, step) if L > step else np.array([L / 2])
        p = shapely.get_coordinates(shapely.line_interpolate_point(ln, d))
        t = (shapely.get_coordinates(shapely.line_interpolate_point(ln, np.clip(d + 1, 0, L)))
             - shapely.get_coordinates(shapely.line_interpolate_point(ln, np.clip(d - 1, 0, L))))
        n = np.linalg.norm(t, axis=1, keepdims=True); n[n == 0] = 1; t = t / n; nrm = np.c_[-t[:, 1], t[:, 0]]
        off = np.full(len(p), np.nan)
        for o in offs:
            b = p + o * nrm
            hit = at(*b.T) & at(*(b + ALONG_PX * t).T) & at(*(b - ALONG_PX * t).T) & np.isnan(off)
            off[hit] = o
        res.append((p, t, nrm, off))
    return res


def exclusive(res, major, names, angle=20.0, radius=18.0):
    """Share of each line's points whose red run is nearer to it than to any other parallel street."""
    P = np.vstack([r[0] for r in res]); T = np.vstack([r[1] for r in res])
    E = np.concatenate([np.full(len(r[0]), i) for i, r in enumerate(res)])
    pen = np.where(major, 0.0, SIDE_STREET_PX); A_ = np.degrees(np.arctan2(T[:, 1], T[:, 0])) % 180
    tree = cKDTree(P); out = np.zeros(len(res))
    for i, (p, t, nrm, off) in enumerate(res):
        hit = ~np.isnan(off)
        if not hit.any(): continue
        c = p[hit] + off[hit, None] * nrm[hit]; own = np.abs(off[hit]) + pen[i]
        a = np.degrees(np.arctan2(t[hit, 1], t[hit, 0])) % 180; ok = np.ones(hit.sum(), bool)
        for k, nb in enumerate(tree.query_ball_point(c, radius)):
            nb = np.asarray(nb); nb = nb[E[nb] != i]
            if len(nb): nb = nb[names[E[nb]] != names[i]]
            if len(nb) == 0: continue
            da = np.abs(A_[nb] - a[k]); nb = nb[np.minimum(da, 180 - da) < angle]
            if len(nb) and (np.linalg.norm(P[nb] - c[k], axis=1) + pen[E[nb]]).min() < own[k] - 0.5: ok[k] = False
        h = hit.copy(); h[hit] = ok; out[i] = h.mean()
    return out


named = (st['name'] != '').values
main = (~st['inset'] & st['onMap']).values & named
ins = st['inset'].values & named
st['red'] = 0.0
res = along(st['geometry'].values[main], warp, REDS_EXT, ACROSS_PX)
st.loc[main, 'red'] = exclusive(res, st['highway'][main].str.contains('primary|secondary|tertiary').values,
                                st['name'].values[main])
st.loc[ins, 'red'] = [np.mean(~np.isnan(r[3])) for r in along(st['geometry'].values[ins], inset_warp, REDS_EXT, ACROSS_INSET_PX)]


def smooth_along(values):
    """Length-weighted mean over same-name, roughly parallel edges within ALONG_STREET."""
    a = shapely.get_coordinates(shapely.get_point(st['geometry'].values, 0)); b = shapely.get_coordinates(shapely.get_point(st['geometry'].values, -1))
    ang = np.degrees(np.arctan2(b[:, 1] - a[:, 1], b[:, 0] - a[:, 0])) % 180
    mid_xy = st[['mx', 'my']].to_numpy(); L = st['len'].to_numpy(); out = np.zeros(len(st))
    for name, idx in st.groupby('name').indices.items():
        if not name: out[idx] = values[idx]; continue
        tree = cKDTree(mid_xy[idx])
        for i in idx:
            nb = idx[tree.query_ball_point(mid_xy[i], ALONG_STREET)]
            da = np.abs(ang[nb] - ang[i]); nb = nb[np.minimum(da, 180 - da) < 20]
            out[i] = (values[nb] * L[nb]).sum() / L[nb].sum()
    return out


st['redSmooth'] = smooth_along(st['red'].to_numpy())
st['l1929'] = named & (st['redSmooth'] >= 0.5)

# Hand corrections, each checked on the scan: the red line runs between two parallel streets a block
# apart and went to the one without the 1929 map's own label. The line moves to the labelled street
# over the same stretch.
HAND = [  # (street that took the line, labelled street on the 1929 map)
    ('West Congress Parkway', 'West Harrison Street'),   # label "Harrison"; Congress had no line
    ('West Leland Avenue', 'West Lawrence Avenue'),       # label "Lawrence", Cicero to Western
]
for wrong, right in HAND:
    w_ = st['l1929'] & (st['name'] == wrong)
    x0_, x1_ = st.loc[w_, 'mx'].min() - 200, st.loc[w_, 'mx'].max() + 200
    to_ = (st['name'] == right) & st['mx'].between(x0_, x1_) & ~st['l1929']
    log.info('Hand correction: %s -> %s, %.1f mi', wrong, right, st.loc[w_, 'len'].sum() / MILE)
    st.loc[w_, 'l1929'] = False
    st.loc[to_, 'l1929'] = True

# Today's routes ------------------------------------------------------------------------------------
rp, ra = [], []
for _, p in route_paths:
    ln = shapely.LineString(p); d = np.arange(0, ln.length, 5.0)
    a = shapely.get_coordinates(shapely.line_interpolate_point(ln, d)); b = shapely.get_coordinates(shapely.line_interpolate_point(ln, np.minimum(d + 5, ln.length)))
    rp.append(a); ra.append(np.degrees(np.arctan2(b[:, 1] - a[:, 1], b[:, 0] - a[:, 0])) % 180)
rp, ra = np.vstack(rp), np.concatenate(ra); rtree = cKDTree(rp)
on_route = np.zeros(len(st))
for k, g in enumerate(st['geometry'].values):
    L = g.length
    if L == 0: continue
    d = np.arange(5, L, 10.0) if L > 10 else np.array([L / 2])
    a = shapely.get_coordinates(shapely.line_interpolate_point(g, d))
    t = shapely.get_coordinates(shapely.line_interpolate_point(g, np.minimum(d + 2, L))) - shapely.get_coordinates(shapely.line_interpolate_point(g, np.maximum(d - 2, 0)))
    ang = np.degrees(np.arctan2(t[:, 1], t[:, 0])) % 180; hit = np.zeros(len(a), bool)
    for i, nb in enumerate(rtree.query_ball_point(a, ON_ROUTE)):
        if nb: da = np.abs(ra[nb] - ang[i]); hit[i] = (np.minimum(da, 180 - da) < 25).any()
    on_route[k] = hit.mean()
st['today'] = on_route >= 0.5

U = st[st['inCity'] & st['onMap'] & ~st['highway'].str.contains('trunk|motorway')].copy()
U['kind'] = np.select([U.l1929 & U.today, U.l1929, U.today], ['both', 'old', 'new'], default='')
log.info('Universe: %d street edges inside the city and on the sheet', len(U))

## 6. Analyze

Street-miles are counted on 40-unit cells so both roadways of a divided street count once; the
edge-length sums (both roadways) are kept as a check.

In [ ]:
def cell_miles(mask) -> float:
    s = set()
    for g in U['geometry'][mask].values:
        p = shapely.get_coordinates(shapely.line_interpolate_point(g, np.arange(0, g.length, 10.0)))
        s.update(map(tuple, np.floor(p / CELL).astype(int)))
    return s


c29, cnow = cell_miles(U.l1929), cell_miles(U.today)
both = c29 & cnow
line29 = shapely.union_all(U['geometry'][U.l1929].values)
U['to1929'] = shapely.distance(U['geometry'].values, line29)
beyond = cell_miles(U.today & ~U.l1929 & (U.to1929 > HALF_MILE))
mi = lambda cells: round(len(cells) * CELL / MILE, 1)
summary = {
    'miles1929': mi(c29), 'milesToday': mi(cnow), 'milesBoth': mi(both),
    'miles1929Only': mi(c29 - cnow), 'milesTodayOnly': mi(cnow - c29), 'milesBeyond': mi(beyond),
    'stillServed': round(len(both) / len(c29), 6), 'onA1929Line': round(len(both) / len(cnow), 6),  # rounded once, for display
    'beyondShare': round(len(beyond) / len(cnow), 6),
    'offSheetMiles': round(float(st.loc[st.inCity & ~st.onMap, 'len'].sum() / MILE), 1),
}
edge = lambda m: float(U.loc[m, 'len'].sum() / MILE)
summary['edgeCheck'] = {'stillServed': round(edge(U.l1929 & U.today) / edge(U.l1929), 3),
                        'onA1929Line': round(edge(U.l1929 & U.today) / edge(U.today), 3)}

# Rings from State & Madison, counted on the same cells as the totals (divided roads once): each cell
# goes to the ring its centre lies in, so the rings add up to the summary.
def ring_of(cells):
    c = np.array(sorted(cells), float) + 0.5
    return np.floor(np.hypot(c[:, 0] * CELL / MILE, y_miles(c[:, 1] * CELL)) / 2).astype(int) * 2
rings = []
ring_cells = {k: ring_of(v) for k, v in (('both', both), ('old', c29 - cnow), ('new', cnow - c29))}
for r0 in range(0, 22, 2):
    f = lambda k: round(float((ring_cells[k] == r0).sum() * CELL / MILE), 1)
    rings.append({'from': r0, 'to': r0 + 2, 'both': f('both'), 'old': f('old'), 'new': f('new')})
assert sum(len(v) for v in ring_cells.values()) == sum(((v >= 0) & (v < 22)).sum() for v in ring_cells.values()), 'a cell lies beyond the last ring'
rings = [r for r in rings if r['both'] + r['old'] + r['new'] > 0]
for r in rings: r['onA1929Line'] = round(r['both'] / max(r['both'] + r['new'], 1e-9), 6)
print(json.dumps(summary, indent=1)); print(pd.DataFrame(rings).to_string(index=False))

top_new = U[U.kind == 'new'].groupby('name')['len'].sum().div(MILE).sort_values(ascending=False).head(12)
top_old = U[U.kind == 'old'].groupby('name')['len'].sum().div(MILE).sort_values(ascending=False).head(12)
print('\nBus today, no 1929 line:\n', top_new.round(1).to_string(), '\n\n1929 line, no bus today:\n', top_old.round(1).to_string())

In [ ]:
# The finding printed on the plate, generated from the data so it can never drift from the numbers.
pct = lambda v: f'{int(v * 100 + 0.5):d}%'   # half up, as the site's toFixed(0) on these values
FINDING = (f'{pct(summary["stillServed"])} of the street-miles of the Chicago Surface Lines (CSL) network on its 1929 map '
           f'still carry a CTA bus. {pct(summary["onA1929Line"])} of CTA bus street-miles today run on a CSL line, and only '
           f'{pct(summary["beyondShare"])} lie more than half a mile from one.')
print(FINDING)

## 7. QA

- **Registration.** Margin RMS and the line-snapping residuals (section 4).
- **Totals.** The 1929 street-miles against the Surface Lines' roughly 1,060 miles of track (a secondary
  figure, double track counted twice), so about 530 street-miles.
- **Sample by eye.** 40 random "today only" and 40 "1929 only" edges drawn on the scan
  (`data/processed/qa_*.png`); the counts found right are entered below and give the stated range.

In [ ]:
SAMPLE_SEED = 6
rng = np.random.default_rng(SAMPLE_SEED)
def sheet(kind, path):
    c = U[(U.kind == kind) & (U.len > 120)]; pick = c.iloc[rng.choice(len(c), 40, replace=False)]
    scan = Image.open(SCAN).convert('RGB'); out = Image.new('RGB', (8 * 244, 5 * 244), 'white')
    from PIL import ImageDraw
    for k, r in enumerate(pick.itertuples()):
        xy = np.asarray(r.geometry.coords); qx, qy = (inset_warp if r.inset else warp)(xy[:, 0], xy[:, 1]); cx, cy = qx.mean(), qy.mean()
        t = scan.crop((int(cx) - 60, int(cy) - 60, int(cx) + 60, int(cy) + 60)).resize((240, 240)); d = ImageDraw.Draw(t)
        d.line(list(zip((qx - cx + 60) * 2, (qy - cy + 60) * 2)), fill=(0, 60, 255), width=3)
        d.rectangle([0, 0, 240, 16], fill=(255, 255, 255)); d.text((3, 2), f'{k + 1}. {r.name[:30]}', fill=(0, 0, 0))
        out.paste(t, ((k % 8) * 244, (k // 8) * 244))
    out.save(path)
sheet('new', PROC_DIR / 'qa_today_only.png'); sheet('old', PROC_DIR / 'qa_1929_only.png')

# Counts found right by eye on the sheets above (re-check whenever the method changes). Checked 2026-11-05,
# after the hand corrections: today only, 1 wrong (Diversey west of Laramie, a dashed 1929 bus line);
# 1929 only, 7 wrong, each a line taken by the next parallel street (74th Pl, 110th Pl, 47th Pl, Edbrooke by
# Michigan, Farrell by Archer, Keating by Cicero, one unclear near Winchester).
QA_RIGHT = {'todayOnly': 39, 'only1929': 33, 'of': 40}
tw = len(cnow - c29) * (1 - QA_RIGHT['todayOnly'] / QA_RIGHT['of'])      # today-only cells likely on a 1929 line
ow = len(c29 - cnow) * (1 - QA_RIGHT['only1929'] / QA_RIGHT['of'])       # 1929-only cells likely no 1929 line
summary['range'] = {
    'onA1929Line': [summary['onA1929Line'], round((len(both) + tw) / len(cnow), 6)],
    'stillServed': [summary['stillServed'], round((len(both) + tw) / (len(c29) + tw - ow), 6)],
}
qa = {
    'margin RMS under 45 units': max(margin_rms.values()) < 45,
    'last snapping pass median offset under 4 px': (snap_log[snap_log.window == SNAP_SCHEDULE[-1][0]].medianOffset < 4).all(),
    '1929 street-miles within 20% of 530': abs(summary['miles1929'] / 530 - 1) < 0.2,
    'cells and edge sums agree within 4 points': abs(summary['onA1929Line'] - summary['edgeCheck']['onA1929Line']) < 0.04,
}
for name, ok in qa.items(): log.info('%-50s %s', name, 'ok' if ok else 'FAILED')
print(summary['range'])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 11))
for kind, color, lw in (('both', 'k', 1.0), ('old', 'tab:red', 1.0), ('new', 'tab:blue', 0.6)):
    gpd.GeoSeries(U.loc[U.kind == kind, 'geometry'].values).plot(ax=ax, color=color, linewidth=lw)
ax.set_aspect('equal'); ax.set_axis_off(); ax.set_title('1929 line and bus today (black), 1929 only (red), today only (blue)')
plt.show()

## 8. Export

`plate.json` carries the ledger, the finding, the summary, the rings, and every street of the
comparison as simplified paths in address units, merged by street name and kind (`both`, `old` = 1929
only, `new` = today only). Keys are camelCase because the website reads them directly.

In [ ]:
def tidy(name: str) -> str:
    """'South Doctor Martin Luther King Junior Drive' -> 'King Dr'; 'West 47th Street' -> '47th St'."""
    name = name.split('|')[0]
    if 'Martin Luther King' in name: return 'King Dr'
    words = name.split()
    if words and words[0] in ('North', 'South', 'East', 'West') and len(words) > 2: words = words[1:]
    abbr = {'Street': 'St', 'Avenue': 'Ave', 'Boulevard': 'Blvd', 'Road': 'Rd', 'Drive': 'Dr', 'Parkway': 'Pkwy', 'Place': 'Pl', 'Court': 'Ct'}
    if words and words[-1] in abbr: words[-1] = abbr[words[-1]]
    return ' '.join(words)


segments = []
for (name, kind), grp in U[U.kind != ''].groupby(['name', 'kind']):
    g = shapely.simplify(shapely.line_merge(shapely.union_all(grp['geometry'].values)), 8)
    for part in (g.geoms if hasattr(g, 'geoms') else [g]):
        if part.length > 20:
            segments.append({'name': tidy(name), 'kind': kind, 'path': np.rint(np.asarray(part.coords)).astype(int).tolist()})

ledger = Ledger(
    source='Chicago Surface Lines, System of Chicago Surface Lines (1929), scan by the University of Minnesota; CTA GTFS; OpenStreetMap',
    dataset_id=SCAN_ID,
    url=SCAN_RECORD,
    pulled=scan_pulled,
    units=f'{summary["miles1929"]:.0f} street-miles of CSL lines (1929 map), {summary["milesToday"]:.0f} of CTA bus routes today',
    method=[
        f'Scan georeferenced to the address grid from its margin street numbers (RMS {margin_rms["x"]:.0f} / {margin_rms["y"]:.0f} '
        'address units), then by snapping OSM arterials to the red lines; Loop from the map\'s Central District inset',
        'A street is a CSL line where red ink runs along it on the 1929 map for at least half its length (smoothed over a quarter mile)',
        f'Today: Plate II\'s CTA bus routes (GTFS {day02["gtfs"]["feedDate"]}) on the street, within {ON_ROUTE} address units',
        f'Street-miles inside today\'s city on {CELL}-unit cells, so both roadways of a divided street count once',
    ],
    caveats=[
        f'A check by eye of 40 + 40 random streets: {QA_RIGHT["only1929"]}/40 "CSL only" and {QA_RIGHT["todayOnly"]}/40 '
        f'"CTA only" right; shares likely {pct(summary["range"]["onA1929Line"][0])}–{pct(summary["range"]["onA1929Line"][1])} and '
        f'{pct(summary["range"]["stillServed"][0])}–{pct(summary["range"]["stillServed"][1])}',
        'The 1929 sheet is the CSL\'s own system map for riders, a snapshot of its network that year; a few CSL bus lines (dashed) may be missed',
        'Two CSL lines moved by hand to the street the map labels: Congress to Harrison, Leland to Lawrence',
        f'{summary["offSheetMiles"]:.0f} miles of city streets lie off the sheet (O\'Hare area) and are left out; expressways and Lake Shore Drive too',
    ],
)
write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload={
    'scan': {'record': SCAN_RECORD, 'iiif': SCAN_IIIF, 'osher': OSHER_RECORD, 'publisher': 'Chicago Surface Lines', 'year': 1929,
             'size': [W, H], 'scale': '1:104,500'},
    'osmDate': osm_date.isoformat(), 'gtfs': day02['gtfs'], 'ctaAccessUrl': CTA_ACCESS_URL,
    'summary': summary, 'rings': rings, 'segments': segments,
    'topNew': [{'name': tidy(n), 'miles': round(v, 1)} for n, v in top_new.items()],
    'topOld': [{'name': tidy(n), 'miles': round(v, 1)} for n, v in top_old.items()],
    'qa': {k: bool(v) for k, v in qa.items()}, 'qaRight': QA_RIGHT,
})
log.info('Wrote %s (%.0f KB): %d segments', OUT_PATH, OUT_PATH.stat().st_size / 1024, len(segments))

## Notes for the plate

- Finding: generated in section 6.
- Design (study 2026-11-05): map A, band and line (1929 = wide gray band, today = thin ink); chart 1 then
  and now, chart 2 by distance from the Loop (`rings`); inset Bronzeville to Washington Park, vector.